# Provision Polaris test catalogs

The event processor's startup self-test (enabled by `CSEP_STARTUP_ICEBERG_SELF_TEST=true`)
writes to a synthetic Polaris catalog `user_event_processor_startup_test`. The manual
integration self-test does the same with `user_event_processor_integration_test`.

Locally, `docker_compose/polaris/polaris_setup.sh` provisions these catalogs automatically.
In **dev / stage / prod**, Polaris catalogs are created on demand by `minio_manager_service`
for real KBase users only — the synthetic test catalogs don't exist, so the self-test fails:

```
RESTException: Unable to process: Unable to find warehouse user_event_processor_startup_test
```

This notebook calls the Polaris management API to create those test catalogs, mirroring what
`polaris_setup.sh` does locally. Idempotent — re-running is safe.

## What this notebook does NOT do

It does not touch MinIO IAM. The event processor's MinIO user must already have bucket-wide
Get / List / Put / Delete on the warehouse bucket (`cdm-lake` in BERDL clusters) — that's a
separate one-time MinIO admin setup. If you see `S3Exception: 400 / 403 AccessDenied` after
running this notebook, check the event processor's attached MinIO policies.

## When to run

* Once per environment (dev, stage, prod) after Polaris is deployed and the event processor's
  MinIO user has write access on the warehouse bucket.
* Re-run if the Polaris persistent volume is wiped.

## Prerequisites

* Polaris management API reachable (`POLARIS_HOST`)
* Polaris bootstrap (root) credentials with permission to create catalogs and grant roles
* The MinIO S3 bucket + warehouse prefix Polaris should write to


## 1. Configuration

**Set the env vars below before running this cell.** The Polaris root client credentials are
required and have no usable defaults; the rest fall back to the in-cluster BERDL defaults.

### MinIO endpoint gotchas

`MINIO_ENDPOINT` gets baked into Polaris's `storageConfigInfo` and is what Iceberg's S3FileIO
(Java AWS SDK v2) connects to for every read / write. Two gotchas verified against dev / stage:

1. MinIO listens HTTPS-ONLY on port 9000 in BERDL clusters. Plain `http://minio.<env>:9000`
   fails with ConnectionClosed, which Spark surfaces as a masked `400 null` /
   `Failed to close current writer`.
2. The in-cluster cert for `minio.<env>:9000` is signed by the cluster's internal CA, not a
   public CA. The Java SDK validates by default and rejects it with the same masked-400
   symptom. Use the EXTERNAL hostname (Let's Encrypt-signed) to avoid the cert-trust problem
   entirely.

Recommended values per environment:

* dev   -> `https://minio.dev.berdl.kbase.us:31276`
* stage -> `https://minio.stage.berdl.kbase.us`
* prod  -> `https://minio.berdl.kbase.us`
* local -> `http://minio:9000` (in-cluster, no TLS, untouched by this issue)


In [ ]:
import os

# ----- Polaris management API -----
# Required: set POLARIS_ROOT_CLIENT_ID and POLARIS_ROOT_CLIENT_SECRET as env vars before
# running this notebook. The "xxxx" defaults intentionally fail fast rather than silently
# using a wrong password.
POLARIS_HOST = os.environ.get("POLARIS_HOST", "http://polaris:8181")
POLARIS_REALM = os.environ.get("POLARIS_REALM", "POLARIS")
POLARIS_ROOT_CLIENT_ID = os.environ.get("POLARIS_ROOT_CLIENT_ID", "xxxx")
POLARIS_ROOT_CLIENT_SECRET = os.environ.get("POLARIS_ROOT_CLIENT_SECRET", "xxxx")

# The Polaris principal role the event processor authenticates as. New catalog roles are
# granted to this principal role so the event processor can write. `service_admin` is
# built-in and bypasses explicit catalog-role binding.
SERVICE_PRINCIPAL_ROLE = os.environ.get("POLARIS_SERVICE_PRINCIPAL_ROLE", "service_admin")

# ----- Where new catalogs land in S3 -----
# See the markdown above for per-environment endpoint guidance.
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
WAREHOUSE_BUCKET = os.environ.get("WAREHOUSE_BUCKET", "cdm-lake")
WAREHOUSE_PREFIX = os.environ.get("WAREHOUSE_PREFIX", "users-sql-warehouse")

# Synthetic test users matching the hardcoded values in cdmsparkevents/selftest/.
TEST_USERS = [
    "event_processor_startup_test",      # cdmsparkevents/selftest/startup.py
    "event_processor_integration_test",  # cdmsparkevents/selftest/integration.py
]

MGMT = f"{POLARIS_HOST.rstrip('/')}/api/management/v1"

print(f"Polaris management API : {MGMT}")
print(f"Realm                  : {POLARIS_REALM}")
print(f"Service principal role : {SERVICE_PRINCIPAL_ROLE}")
print(f"S3 endpoint            : {MINIO_ENDPOINT}")
print(f"Warehouse              : s3://{WAREHOUSE_BUCKET}/{WAREHOUSE_PREFIX}/<user>/iceberg/")
print(f"Test users to provision: {TEST_USERS}")


## 2. Get an OAuth token from Polaris

Uses the root client credentials to mint a token with `PRINCIPAL_ROLE:ALL` scope. The token
is short-lived (1 hour by default); re-run this cell if subsequent cells start returning 401s.


In [ ]:
import requests


def get_polaris_token() -> str:
    resp = requests.post(
        f"{POLARIS_HOST.rstrip('/')}/api/catalog/v1/oauth/tokens",
        auth=(POLARIS_ROOT_CLIENT_ID, POLARIS_ROOT_CLIENT_SECRET),
        headers={"Polaris-Realm": POLARIS_REALM},
        data={"grant_type": "client_credentials", "scope": "PRINCIPAL_ROLE:ALL"},
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


TOKEN = get_polaris_token()
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Polaris-Realm": POLARIS_REALM,
}
print(f"Got Polaris token ({len(TOKEN)} chars)")


## 3. Provision a catalog (helper)

For each test user this:

1. Creates a Polaris catalog `user_<name>` of type `INTERNAL` with `storageConfigInfo`
   pointing at `s3://<WAREHOUSE_BUCKET>/<WAREHOUSE_PREFIX>/<name>/iceberg/`.
2. Creates a catalog role `user_<name>_admin` with `CATALOG_MANAGE_CONTENT` privilege.
3. Grants that catalog role to the service principal role so the event processor can write.

Each step treats `409 Conflict` as a no-op (already exists) so the notebook is idempotent.


In [ ]:
def _check(resp: requests.Response, ok_statuses=(200, 201, 204), tolerated=(409,)) -> None:
    """Treat 200/201/204 as success, 409 as idempotent skip, anything else as fatal."""
    if resp.status_code in ok_statuses:
        print(f"    -> {resp.status_code} ok")
    elif resp.status_code in tolerated:
        print(f"    -> {resp.status_code} already exists (skipped)")
    else:
        raise RuntimeError(
            f"Polaris call failed ({resp.status_code}): {resp.text[:300]}"
        )


def provision_catalog(user: str) -> None:
    catalog = f"user_{user}"
    role = f"{catalog}_admin"
    location = f"s3://{WAREHOUSE_BUCKET}/{WAREHOUSE_PREFIX}/{user}/iceberg/"
    print(f"\nProvisioning {catalog}")
    print(f"  location: {location}")

    print("  - creating catalog")
    _check(requests.post(
        f"{MGMT}/catalogs",
        headers=HEADERS,
        json={
            "catalog": {
                "name": catalog,
                "type": "INTERNAL",
                "properties": {"default-base-location": location},
                "storageConfigInfo": {
                    "storageType": "S3",
                    "allowedLocations": [location],
                    "endpoint": MINIO_ENDPOINT,
                    "endpointInternal": MINIO_ENDPOINT,
                    "pathStyleAccess": True,
                    "stsUnavailable": True,
                    "region": "us-east-1",
                },
            }
        },
        timeout=10,
    ))

    print("  - creating catalog role")
    _check(requests.post(
        f"{MGMT}/catalogs/{catalog}/catalog-roles",
        headers=HEADERS,
        json={"catalogRole": {"name": role}},
        timeout=10,
    ))

    print("  - applying CATALOG_MANAGE_CONTENT grant to catalog role")
    # Polaris returns 500 instead of 409 when the grant already exists; tolerate both.
    _check(
        requests.put(
            f"{MGMT}/catalogs/{catalog}/catalog-roles/{role}/grants",
            headers=HEADERS,
            json={"grant": {"type": "catalog", "privilege": "CATALOG_MANAGE_CONTENT"}},
            timeout=10,
        ),
        tolerated=(409, 500),
    )

    if SERVICE_PRINCIPAL_ROLE != "service_admin":
        # service_admin is built-in and already has full access; only bind the catalog
        # role to a custom principal role.
        print(f"  - binding catalog role to principal role {SERVICE_PRINCIPAL_ROLE}")
        _check(requests.put(
            f"{MGMT}/principal-roles/{SERVICE_PRINCIPAL_ROLE}/catalog-roles/{catalog}",
            headers=HEADERS,
            json={"catalogRole": {"name": role}},
            timeout=10,
        ))
    else:
        print("  - service_admin role bypasses explicit catalog-role binding")


## 4. Run the provisioning loop


In [ ]:
for user in TEST_USERS:
    provision_catalog(user)

print("\nProvisioning complete.")


## 5. Verify

List the catalogs Polaris knows about and confirm each `user_<name>` is present.


In [ ]:
resp = requests.get(f"{MGMT}/catalogs", headers=HEADERS, timeout=10)
resp.raise_for_status()
all_catalogs = sorted(c["name"] for c in resp.json().get("catalogs", []))
expected = sorted(f"user_{u}" for u in TEST_USERS)
missing = [c for c in expected if c not in all_catalogs]
extra_user_catalogs = [c for c in all_catalogs if c.startswith("user_") and c not in expected]

print("Expected (this notebook):")
for c in expected:
    mark = "OK " if c in all_catalogs else "MISSING"
    print(f"  [{mark}] {c}")

if extra_user_catalogs:
    print("\nOther user_* catalogs already present in Polaris (likely real KBase users from MMS):")
    for c in extra_user_catalogs:
        print(f"  - {c}")

if missing:
    raise RuntimeError(f"Provisioning failed for: {missing}")
print("\nAll expected catalogs present.")


## 6. Cleanup (optional)

Drops every catalog this notebook would create. Run only if you want to remove the synthetic
test catalogs.

**Warning:** Polaris won't drop a non-empty catalog, so you'll get errors here if data has
been written to a synthetic test catalog — clean those out first via Spark.


In [ ]:
# Uncomment to drop the test catalogs.
# for user in TEST_USERS:
#     catalog = f"user_{user}"
#     print(f"Deleting {catalog}")
#     resp = requests.delete(f"{MGMT}/catalogs/{catalog}", headers=HEADERS, timeout=10)
#     if resp.status_code in (200, 204):
#         print(f"    -> deleted")
#     elif resp.status_code == 404:
#         print(f"    -> 404 not found (already gone)")
#     else:
#         print(f"    -> {resp.status_code}: {resp.text[:300]}")
